# LLM-judge mean score charts (gold / noise separately)

Single data source: `results/judge_score_bootstrap_ci.json`.
Rebuild from repository root: `uv run python scripts/build_judge_score_artifacts.py`.

Both bar height (mean) and error bars (95% percentile bootstrap CI) come from one file —
methodological consistency is guaranteed.

2×2 panels: X-axis — **models**, **two bars** per group (baseline and full); each model has a base color, two shades within the group (muted → bright). Legend — modes (shade sample from the middle model).

PNG (dpi=300) saved to `charts/`: Russian — `judge_scores_gold_bootstrap_panel.png`, `judge_scores_noise_bootstrap_panel.png`, `judge_scores_gold_noise_combined_panel.png`. English combined panels (per mode) — `judge_scores_gold_noise_combined_{baseline,full}_panel_en.png`.


In [ ]:
from __future__ import annotations

import json
import math
from dataclasses import dataclass
from pathlib import Path
from typing import Callable

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch


@dataclass(frozen=True)
class ChartFontSizes:
    tick: int = 12
    axis_label: int = 14
    subplot_title: int = 15
    suptitle: int = 18
    legend: int = 12
    legend_title: int = 13


MODELS: tuple[str, ...] = ("gemma_4", "nemotron", "qwen_35")
MODES: tuple[str, ...] = ("baseline", "full")
MODEL_DISPLAY_NAMES: dict[str, str] = {
    "gemma_4": "RedHatAI/gemma-4-31B-it-NVFP4",
    "nemotron": "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-NVFP4",
    "qwen_35": "Qwen/Qwen3.5-35B-A3B-GPTQ-Int4",
}

# Metric keys match the "metric" field in judge_score_bootstrap_ci.json.
METRICS_RU: tuple[tuple[str, str], ...] = (
    ("relevance", "Релевантность"),
    ("correctness", "Корректность"),
    ("faithfulness", "Достоверность"),
    ("completeness", "Полнота"),
)
METRICS_EN: tuple[tuple[str, str], ...] = (
    ("relevance", "Relevance"),
    ("correctness", "Correctness"),
    ("faithfulness", "Faithfulness"),
    ("completeness", "Completeness"),
)
CLUSTER_BASE_COLORS: tuple[str, ...] = ("#2a6f97", "#c1666b", "#6a994e")
# Tableau 10 palette (first three) — industry-standard for data visualisation.
CLUSTER_BASE_COLORS_RICH: tuple[str, ...] = ("#4e79a7", "#f28e2b", "#59a14f")
FONT_SIZES = ChartFontSizes()
BOOTSTRAP_CI_FILENAME = "judge_score_bootstrap_ci.json"

# (model, bench, mode, metric) → (mean, ci_low, ci_high)
BootstrapLookup = dict[tuple[str, str, str, str], tuple[float, float, float]]


def find_repository_root() -> Path:
    """Find repository root by presence of results/judge_score_bootstrap_ci.json."""
    here = Path.cwd().resolve()
    for directory in (here, *here.parents):
        if (directory / "results" / BOOTSTRAP_CI_FILENAME).is_file():
            return directory
    raise FileNotFoundError(
        f"Missing results/{BOOTSTRAP_CI_FILENAME} "
        "— open the notebook from the AI_assistant_benchs_public repository root."
    )


def load_bootstrap_ci(path: Path) -> BootstrapLookup:
    """Load bootstrap_ci.json into a dict; records with nan are skipped."""
    raw = json.loads(path.read_text(encoding="utf-8"))
    lookup: BootstrapLookup = {}
    for row in raw:
        try:
            key = (str(row["model"]), str(row["bench"]), str(row["mode"]), str(row["metric"]))
            triple = (float(row["mean"]), float(row["ci_low"]), float(row["ci_high"]))
        except (KeyError, TypeError, ValueError):
            continue
        if all(math.isfinite(v) for v in triple):
            lookup[key] = triple
    return lookup


def model_legend_label(model: str) -> str:
    """Short model label for per-bench x-axis ticks."""
    parts = [piece for piece in model.replace("_", " ").split() if not piece.isdigit()]
    return " ".join(parts) if parts else model.replace("_", " ")


def model_display_name(model: str) -> str:
    """Full model identifier for combined-panel legend."""
    return MODEL_DISPLAY_NAMES.get(model, model)


def shades_for_cluster(hex_base: str, modes: tuple[str, ...]) -> tuple[str, ...]:
    """Shades of one color: muted to saturated; number of steps = len(modes)."""
    base = np.asarray(mcolors.to_rgb(hex_base), dtype=np.float64)
    num_modes = len(modes)
    if num_modes <= 0:
        return ()
    if num_modes == 1:
        return (mcolors.rgb2hex(tuple(np.clip(base, 0.0, 1.0))),)
    dim = np.clip(
        0.58 * base + 0.42 * np.array([0.78, 0.78, 0.80], dtype=np.float64),
        0.0,
        1.0,
    )
    hex_colors: list[str] = []
    for step_index in range(num_modes):
        blend = step_index / (num_modes - 1)
        rgb = np.clip((1.0 - blend) * dim + blend * base, 0.0, 1.0)
        hex_colors.append(mcolors.rgb2hex(tuple(rgb)))
    return tuple(hex_colors)


def fixed_score_axis_limits() -> tuple[float, float]:
    """Fixed Y-axis scale 4.5–5.0 for bar charts (scores 1–5)."""
    return 4.5, 5.0




In [ ]:
class JudgeScoresBarCharts:
    """Load bootstrap CI and build chart panels.

    Single data source — judge_score_bootstrap_ci.json.
    Both bar height (mean) and error bars (ci_low, ci_high) come
    from the same record fields — no methodology mismatch.
    """

    def __init__(
        self,
        *,
        project_root: Path,
        ci_lookup: BootstrapLookup,
        font_sizes: ChartFontSizes = FONT_SIZES,
    ) -> None:
        self._project_root = project_root.resolve()
        self._ci = ci_lookup
        self._font_sizes = font_sizes
        self._charts_dir = self._project_root / "charts"
        self._charts_dir.mkdir(parents=True, exist_ok=True)

    @classmethod
    def from_repository(cls) -> JudgeScoresBarCharts:
        root = find_repository_root()
        path = root / "results" / BOOTSTRAP_CI_FILENAME
        ci_lookup = load_bootstrap_ci(path)
        print("JSON:", path)
        print("Records:", len(ci_lookup))
        return cls(project_root=root, ci_lookup=ci_lookup)

    def _entry(self, model: str, bench: str, mode: str, metric: str) -> tuple[float, float, float] | None:
        """Triple (mean, ci_low, ci_high) from lookup or None."""
        return self._ci.get((model, bench, mode, metric))

    def plot_bench_four_figures_panel(
        self,
        *,
        title_prefix: str,
        bench: str,
        file_stem: str,
        metrics: tuple[tuple[str, str], ...] = METRICS_RU,
        xlabel: str = "Модель",
        ylabel: str = "Средняя оценка (1–5)",
        legend_title: str = "Режим",
    ) -> None:
        """2×2 figure: four metrics, X-axis — models, two modes per model group."""
        x_positions = np.arange(len(MODELS))
        num_modes = len(MODES)
        bar_width = 0.22
        offsets = np.linspace(
            -(num_modes - 1) / 2 * bar_width,
            (num_modes - 1) / 2 * bar_width,
            num_modes,
        )
        band_half_width = bar_width * 0.58
        axis_min, axis_max = fixed_score_axis_limits()

        figure, axes = plt.subplots(2, 2, figsize=(14.5, 10.5))
        axes_flat = axes.flatten()
        for axis_index, (metric, metric_label) in enumerate(metrics):
            axis = axes_flat[axis_index]
            for mode_index, mode in enumerate(MODES):
                offset = offsets[mode_index]
                bar_heights = [
                    entry[0] if (entry := self._entry(model, bench, mode, metric)) is not None else float("nan")
                    for model in MODELS
                ]
                bar_colors = [
                    shades_for_cluster(CLUSTER_BASE_COLORS[mi], MODES)[mode_index]
                    for mi in range(len(MODELS))
                ]
                axis.bar(
                    x_positions + offset,
                    bar_heights,
                    bar_width,
                    color=bar_colors,
                    edgecolor="white",
                    linewidth=0.6,
                    zorder=2,
                )
            for mode_index, mode in enumerate(MODES):
                offset = offsets[mode_index]
                for mi, model in enumerate(MODELS):
                    entry = self._entry(model, bench, mode, metric)
                    if entry is None:
                        continue
                    _, low, high = entry
                    x_center = float(x_positions[mi] + offset)
                    face_color = shades_for_cluster(CLUSTER_BASE_COLORS[mi], MODES)[mode_index]
                    axis.fill_betweenx(
                        [low, high],
                        x_center - band_half_width,
                        x_center + band_half_width,
                        facecolor=face_color,
                        edgecolor="white",
                        linewidth=0.9,
                        alpha=0.5,
                        zorder=6,
                    )
            axis.set_xticks(x_positions, [model_legend_label(m) for m in MODELS])
            axis.tick_params(axis="both", which="major", labelsize=self._font_sizes.tick)
            axis.set_xlabel(xlabel, fontsize=self._font_sizes.axis_label)
            axis.set_ylabel(ylabel, fontsize=self._font_sizes.axis_label)
            axis.set_title(metric_label, fontsize=self._font_sizes.subplot_title)
            axis.set_ylim(axis_min, axis_max)
            axis.grid(axis="y", alpha=0.35)
            axis.set_axisbelow(True)

        legend_model_index = len(MODELS) // 2
        legend_handles = [
            Patch(
                facecolor=shades_for_cluster(CLUSTER_BASE_COLORS[legend_model_index], MODES)[mi],
                edgecolor="white",
                linewidth=0.6,
                label=MODES[mi].capitalize(),
            )
            for mi in range(len(MODES))
        ]
        if title_prefix.strip():
            figure.suptitle(title_prefix, fontsize=self._font_sizes.suptitle, y=1.02)
        figure.legend(
            handles=legend_handles,
            title=legend_title,
            ncol=len(MODES),
            loc="lower center",
            bbox_to_anchor=(0.5, -0.06),
            framealpha=0.95,
            fontsize=self._font_sizes.legend,
            title_fontsize=self._font_sizes.legend_title,
        )
        top_margin = 0.96 if title_prefix.strip() else 1.0
        figure.tight_layout(rect=(0.0, 0.11, 1.0, top_margin))
        output_path = self._charts_dir / f"{file_stem}_panel.png"
        figure.savefig(output_path, dpi=300, bbox_inches="tight")
        plt.show()
        plt.close(figure)
        print("Saved:", output_path)

    def plot_bench_two_benches_panel_four_figures(
        self,
        *,
        title_prefix: str,
        file_stem: str,
        mode: str = "full",
        metrics: tuple[tuple[str, str], ...] = METRICS_RU,
        bench_tick_labels: tuple[str, str] = ("Без шума", "С шумом"),
        xlabel: str = "Бенчмарк",
        ylabel: str = "Средняя оценка (1–5)",
        legend_title: str = "Модель",
        colors: tuple[str, ...] = CLUSTER_BASE_COLORS,
        model_label_fn: Callable[[str], str] = model_legend_label,
        legend_ncol: int = 3,
        legend_fontsize: int | None = None,
        figure_size: tuple[float, float] = (13.0, 10.0),
        layout_bottom: float = 0.10,
    ) -> None:
        """2×2 figure: gold and noise side by side, one mode, X-axis — bench type."""
        if mode not in MODES:
            raise ValueError(f"Unknown mode {mode!r}; expected one of {MODES}")
        benches = ("gold", "noise")
        bench_centers = np.arange(len(benches), dtype=float)
        num_models = len(MODELS)
        bar_width = 0.22
        offsets = np.linspace(
            -(num_models - 1) / 2 * bar_width,
            (num_models - 1) / 2 * bar_width,
            num_models,
        )
        band_half_width = bar_width * 0.58
        axis_min, axis_max = 4.6, 5.0

        figure, axes = plt.subplots(2, 2, figsize=figure_size)
        axes_flat = axes.ravel()
        legend_handles_labels: tuple | None = None

        for panel_index, (axis, (metric, metric_label)) in enumerate(zip(axes_flat, metrics)):
            for bench_index, bench in enumerate(benches):
                center = float(bench_centers[bench_index])
                for offset, model, color in zip(offsets, MODELS, colors):
                    entry = self._entry(model, bench, mode, metric)
                    bar_height = entry[0] if entry is not None else float("nan")
                    label = model_label_fn(model) if bench_index == 0 else "_nolegend_"
                    axis.bar(
                        [center + offset],
                        [bar_height],
                        bar_width,
                        label=label,
                        color=color,
                        edgecolor="white",
                        linewidth=0.6,
                        zorder=2,
                    )
            for bench_index, bench in enumerate(benches):
                center = float(bench_centers[bench_index])
                for offset, model, color in zip(offsets, MODELS, colors):
                    entry = self._entry(model, bench, mode, metric)
                    if entry is None:
                        continue
                    _, low, high = entry
                    x_center = center + offset
                    axis.fill_betweenx(
                        [low, high],
                        x_center - band_half_width,
                        x_center + band_half_width,
                        facecolor=color,
                        edgecolor="white",
                        linewidth=0.9,
                        alpha=0.5,
                        zorder=6,
                    )
            if legend_handles_labels is None:
                legend_handles_labels = axis.get_legend_handles_labels()
            axis.set_xticks(bench_centers, bench_tick_labels)
            axis.tick_params(axis="both", which="major", labelsize=self._font_sizes.tick)
            if panel_index >= 2:
                axis.set_xlabel(xlabel, fontsize=self._font_sizes.axis_label)
            if panel_index % 2 == 0:
                axis.set_ylabel(ylabel, fontsize=self._font_sizes.axis_label)
            axis.set_title(metric_label, fontsize=self._font_sizes.subplot_title)
            axis.set_ylim(axis_min, axis_max)
            axis.grid(axis="y", alpha=0.35)
            axis.set_axisbelow(True)

        if legend_handles_labels is not None:
            handles, labels = legend_handles_labels
            figure.legend(
                handles,
                labels,
                title=legend_title,
                ncol=legend_ncol,
                loc="lower center",
                bbox_to_anchor=(0.5, 0.02 if legend_ncol > 1 else 0.0),
                bbox_transform=figure.transFigure,
                framealpha=0.92,
                fontsize=legend_fontsize or self._font_sizes.legend,
                title_fontsize=self._font_sizes.legend_title,
            )
        if title_prefix.strip():
            figure.suptitle(title_prefix, fontsize=self._font_sizes.suptitle, y=1.01)
        figure.tight_layout(rect=(0.0, layout_bottom if legend_ncol > 1 else 0.16, 1.0, 0.96))
        output_path = self._charts_dir / f"{file_stem}.png"
        figure.savefig(output_path, dpi=300, bbox_inches="tight")
        plt.show()
        plt.close(figure)
        print("Saved:", output_path)


charts = JudgeScoresBarCharts.from_repository()


In [ ]:
charts.plot_bench_four_figures_panel(
    title_prefix="Бенчмарк без шума",
    bench="gold",
    file_stem="judge_scores_gold_bootstrap",
)


In [ ]:
charts.plot_bench_four_figures_panel(
    title_prefix="Бенчмарк с шумом",
    bench="noise",
    file_stem="judge_scores_noise_bootstrap",
)


In [ ]:
charts.plot_bench_two_benches_panel_four_figures(
    title_prefix="",
    file_stem="judge_scores_gold_noise_combined_panel",
)


In [ ]:
for rag_mode in MODES:
    charts.plot_bench_two_benches_panel_four_figures(
        title_prefix=f"LLM Judge Scores — {rag_mode.capitalize()} mode",
        file_stem=f"judge_scores_gold_noise_combined_{rag_mode}_panel_en",
        mode=rag_mode,
        metrics=METRICS_EN,
        bench_tick_labels=("Without Noise", "With Noise"),
        xlabel="Benchmark",
        ylabel="Mean Score (1–5)",
        legend_title="Model",
        colors=CLUSTER_BASE_COLORS_RICH,
        model_label_fn=model_display_name,
        legend_ncol=1,
        legend_fontsize=10,
        figure_size=(13.0, 10.5),
        layout_bottom=0.16,
    )